# Proyecto FIFA World Cup 2022  
## F3_Definicion.ipynb — Sumativa 3

**Asignatura:** MCDI500 — Herramientas de software científico  
**Proyecto:** Análisis del Mundial FIFA Qatar 2022  
**Entregable:** Notebook ejecutable F3 con preprocesamiento completo, algoritmos, eficiencia y POO  

> Versión incremental para Pull Request. Versión final integrada: secciones 1 a 22, incluyendo POO, documentación, repo, conclusiones, checklist y plan de PR.

## Índice

1. Contexto y propósito de la Fase 3  
2. Relación con Fase 2 y definición del problema algorítmico  
3. Configuración del entorno  
4. Carga del dataset y preprocesamiento funcional  
4.1. Pipeline completo reutilizado desde F2  
5. Validación del dataset preprocesado  
6. Núcleo algorítmico estructurado  
7. Algoritmo iterativo  
8. Algoritmo recursivo  
9. Algoritmo vectorizado con pandas  
10. Validación técnica del código  
11. Medición de eficiencia y análisis Big-O  
12. Diseño estructurado, cohesión y bajo acoplamiento  
13. Algoritmo recursivo adicional: Merge Sort  
14. Programación Orientada a Objetos  
15. Pipeline POO con herencia, polimorfismo y encapsulamiento  
16. Detección algorítmica de partidos destacados  
17. Evidencia final de ejecución reproducible  
18. Instrucciones para repositorio GitHub  
19. Conclusiones  
20. Bibliografía  
21. Plan de integración mediante Pull Requests  

## 1. Contexto y propósito de la Fase 3

La Fase 3 toma como base el trabajo realizado en la Fase 2, donde se construyó un pipeline de obtención, limpieza, transformación y validación del dataset del Mundial FIFA Qatar 2022.

En esta etapa se desarrolla el núcleo algorítmico del proyecto. El propósito es demostrar que el análisis no solo permite explorar datos, sino también construir soluciones computacionales organizadas, medibles y reutilizables.

La guía de la Sumativa 3 solicita integrar programación estructurada, recursividad, eficiencia algorítmica, mediciones con `timeit`, Programación Orientada a Objetos, documentación técnica y un notebook ejecutable.


## 2. Relación con Fase 2 y definición del problema algorítmico

En la Fase 2 el dataset fue limpiado y transformado para dejar una base reproducible. En esta Fase 3 se reutiliza ese flujo, pero se reorganiza como arquitectura algorítmica.

**Problema algorítmico definido:** calcular y comparar el total de goles registrados en los partidos del Mundial FIFA 2022 utilizando tres enfoques:

1. Un algoritmo estructurado iterativo.
2. Un algoritmo recursivo.
3. Una operación vectorizada con pandas.

Además, se incorpora un algoritmo de ordenamiento recursivo `merge sort` para ordenar partidos según cantidad total de goles y una estructura POO para encapsular el preprocesamiento y los algoritmos de análisis.

Esta elección es pertinente porque la pregunta del proyecto se relaciona con el rendimiento de las selecciones y el comportamiento estadístico de los partidos. Los goles son una variable central para analizar resultados y desempeño.


## 3. Configuración del entorno

Se importan las librerías necesarias. El notebook está preparado para ejecutarse tanto desde la raíz del repositorio como desde una carpeta `notebooks/` o `F3/`. También incluye una carga alternativa con datos mínimos de ejemplo para que el notebook siga siendo ejecutable cuando el CSV original no esté disponible en el entorno de revisión.


In [1]:
from pathlib import Path
from abc import ABC, abstractmethod
import sys
import timeit
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
np.random.seed(42)

# Permite importar módulos del repositorio cuando el notebook se ejecuta desde notebooks/
# o desde la raíz del proyecto.
for ruta_src in [Path("../src"), Path("src"), Path(".")]:
    if ruta_src.exists() and str(ruta_src.resolve()) not in sys.path:
        sys.path.insert(0, str(ruta_src.resolve()))

try:
    import funciones_reutilizables as f2_utils
    import funciones_f3_algoritmos as f3_algos
    MODULOS_REPO_DISPONIBLES = True
    print("Módulos del repositorio importados correctamente desde src/.")
except Exception as exc:
    MODULOS_REPO_DISPONIBLES = False
    f2_utils = None
    f3_algos = None
    print("No se pudieron importar módulos desde src/. Se usarán funciones internas del notebook.")
    print("Detalle:", exc)

print("Entorno configurado correctamente")


Módulos del repositorio importados correctamente desde src/.
Entorno configurado correctamente


## 4. Carga del dataset y preprocesamiento funcional

Esta sección mantiene un flujo funcional de entrada → proceso → salida. Primero se carga el dataset original. Luego se ejecuta un preprocesamiento funcional mínimo para el núcleo algorítmico de goles, manteniendo columnas interpretables como `team1`, `team2` y goles.

En la sección **4.1** se ejecuta explícitamente el **pipeline completo reutilizado desde F2**, incluyendo manejo de nulos, casting, variables derivadas, codificación categórica, normalización y validación final. Esto responde directamente al feedback de la Formativa 3.


In [2]:
def crear_dataset_demo() -> pd.DataFrame:
    """Crea un dataset mínimo de respaldo para asegurar ejecución reproducible.

    Este respaldo no reemplaza el dataset real; solo permite que el notebook
    ejecute completo si el archivo CSV no está disponible en el entorno.
    """
    return pd.DataFrame({
        "team1": ["QATAR", "ENGLAND", "ARGENTINA", "FRANCE", "BRAZIL", "SPAIN"],
        "team2": ["ECUADOR", "IRAN", "SAUDI ARABIA", "AUSTRALIA", "SERBIA", "COSTA RICA"],
        "number of goals team1": [0, 6, 1, 4, 2, 7],
        "number of goals team2": [2, 2, 2, 1, 0, 0],
        "possession team1": ["47%", "78%", "69%", "63%", "59%", "82%"],
        "possession team2": ["40%", "13%", "20%", "25%", "24%", "9%"],
        "possession in contest": ["13%", "9%", "11%", "12%", "17%", "9%"],
        "total attempts team1": [5, 13, 14, 23, 24, 17],
        "total attempts team2": [6, 8, 3, 4, 4, 0],
        "passes team1": [450, 798, 596, 734, 602, 1045],
        "passes team2": [480, 215, 268, 466, 403, 231],
        "category": ["Group A", "Group B", "Group C", "Group D", "Group G", "Group E"]
    })


def cargar_datos(rutas_posibles: list[Path]) -> pd.DataFrame:
    """Carga el dataset FIFA desde la primera ruta disponible.

    Parámetros
    ----------
    rutas_posibles : list[Path]
        Lista ordenada de rutas candidatas para ubicar el CSV.

    Retorna
    -------
    pd.DataFrame
        Dataset cargado desde CSV o dataset de respaldo si el archivo no existe.
    """
    for ruta in rutas_posibles:
        if ruta.exists():
            df = pd.read_csv(ruta)
            print(f"Archivo utilizado: {ruta}")
            print(f"Dataset cargado: {df.shape[0]} filas y {df.shape[1]} columnas")
            return df

    print("No se encontró el CSV original. Se usará dataset demo para mantener la ejecución reproducible.")
    df = crear_dataset_demo()
    print(f"Dataset demo cargado: {df.shape[0]} filas y {df.shape[1]} columnas")
    return df


def normalizar_nombres_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nombres de columnas a minúsculas, sin espacios extremos."""
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df


def convertir_porcentaje(df: pd.DataFrame, columna: str) -> pd.DataFrame:
    """Convierte una columna porcentual tipo '55%' a valor numérico float.

    Se mantiene el valor en escala 0-100 porque facilita la interpretación
    deportiva de posesión como porcentaje.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame")

    df = df.copy()
    df[columna] = (
        df[columna]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace({"nan": np.nan, "None": np.nan})
        .astype(float)
    )
    return df


def crear_variables_resultado(df: pd.DataFrame) -> pd.DataFrame:
    """Crea variables derivadas asociadas al resultado del partido.

    Se calcula el total de goles, la diferencia de goles del equipo 1
    y una etiqueta simple de resultado para team1.
    """
    requeridas = ["number of goals team1", "number of goals team2"]
    faltantes = [c for c in requeridas if c not in df.columns]
    if faltantes:
        raise KeyError(f"Columnas faltantes para crear resultado: {faltantes}")

    df = df.copy()
    df["total_goals"] = df["number of goals team1"] + df["number of goals team2"]
    df["goal_difference_team1"] = df["number of goals team1"] - df["number of goals team2"]
    df["result_team1"] = np.select(
        [df["goal_difference_team1"] > 0, df["goal_difference_team1"] < 0],
        ["win", "loss"],
        default="draw"
    )
    return df


def imputar_nulos_numericos(df: pd.DataFrame, estrategia: str = "mediana") -> pd.DataFrame:
    """Imputa nulos en columnas numéricas usando media o mediana.

    Decisión técnica: se usa mediana por defecto porque es más robusta
    frente a valores extremos, algo esperable en estadísticas deportivas.
    """
    if estrategia not in {"media", "mediana"}:
        raise ValueError("La estrategia debe ser 'media' o 'mediana'")

    df = df.copy()
    numericas = df.select_dtypes(include=[np.number]).columns
    for col in numericas:
        if df[col].isnull().any():
            valor = df[col].median() if estrategia == "mediana" else df[col].mean()
            df[col] = df[col].fillna(valor)
    return df


def escalar_zscore(df: pd.DataFrame, columnas: list[str]) -> pd.DataFrame:
    """Estandariza columnas numéricas con z-score.

    Decisión técnica: z-score permite comparar variables con distintas escalas,
    por ejemplo pases, intentos y posesión, manteniendo media cercana a 0.
    """
    df = df.copy()
    for col in columnas:
        if col not in df.columns:
            raise KeyError(f"La columna '{col}' no existe")
        if not pd.api.types.is_numeric_dtype(df[col]):
            raise TypeError(f"La columna '{col}' debe ser numérica para escalar")
        std = df[col].std(ddof=0)
        if std == 0 or np.isnan(std):
            df[col] = 0.0
        else:
            df[col] = (df[col] - df[col].mean()) / std
    return df


def preprocesar_datos(df: pd.DataFrame) -> pd.DataFrame:
    """Ejecuta el pipeline funcional de preprocesamiento de Fase 3."""
    df = normalizar_nombres_columnas(df)

    columnas_porcentaje = [
        "possession team1",
        "possession team2",
        "possession in contest"
    ]
    for col in columnas_porcentaje:
        if col in df.columns:
            df = convertir_porcentaje(df, col)

    df = crear_variables_resultado(df)
    df = imputar_nulos_numericos(df, estrategia="mediana")

    columnas_a_escalar = [
        c for c in [
            "possession team1", "possession team2", "possession in contest",
            "total attempts team1", "total attempts team2",
            "passes team1", "passes team2"
        ]
        if c in df.columns
    ]
    df = escalar_zscore(df, columnas_a_escalar)
    return df


def preparar_lista_goles(df: pd.DataFrame) -> list[int]:
    """Convierte las columnas de goles de ambos equipos en una lista plana."""
    columnas = ["number of goals team1", "number of goals team2"]
    faltantes = [c for c in columnas if c not in df.columns]
    if faltantes:
        raise KeyError(f"Faltan columnas de goles: {faltantes}")

    goles = df[columnas].to_numpy().ravel().astype(int).tolist()
    return goles


In [3]:
rutas_posibles = [
    Path("../data/raw/Fifa_world_cup_matches.csv"),
    Path("data/raw/Fifa_world_cup_matches.csv"),
    Path("../F2/data/raw/Fifa_world_cup_matches.csv"),
    Path("F2/data/raw/Fifa_world_cup_matches.csv"),
    Path("/mnt/data/Fifa_world_cup_matches.csv")
]

df_original = cargar_datos(rutas_posibles)
df = preprocesar_datos(df_original)
goles = preparar_lista_goles(df)

print("Dataset original:", df_original.shape)
print("Dataset preprocesado:", df.shape)
print("Cantidad de registros de goles:", len(goles))
print("Primeros 10 goles:", goles[:10])
df.head()


Archivo utilizado: ../data/raw/Fifa_world_cup_matches.csv
Dataset cargado: 64 filas y 88 columnas
Dataset original: (64, 88)
Dataset preprocesado: (64, 91)
Cantidad de registros de goles: 128
Primeros 10 goles: [0, 2, 6, 2, 0, 2, 1, 1, 1, 2]


,team1,team2,possession team1,possession team2,possession in contest,number of goals team1,number of goals team2,date,hour,category,...,goal preventions team2,own goals team1,own goals team2,forced turnovers team1,forced turnovers team2,defensive pressures applied team1,defensive pressures applied team2,total_goals,goal_difference_team1,result_team1
0,QATAR,ECUADOR,-0.222014,0.543986,-1.643741,0,2,20 NOV 2022,17 : 00,Group A,...,5,0,0,52,72,256,279,2,-2,loss
1,ENGLAND,IRAN,2.315290,-2.062941,-1.222943,6,2,21 NOV 2022,14 : 00,Group B,...,13,0,0,63,72,139,416,8,4,win
2,SENEGAL,NETHERLANDS,-0.052861,0.123514,-0.381348,0,2,21 NOV 2022,17 : 00,Group A,...,15,0,0,63,73,263,251,2,-2,loss
3,UNITED STATES,WALES,0.539177,-0.381053,-0.802146,1,1,21 NOV 2022,20 : 00,Group B,...,7,0,0,81,72,242,292,2,0,draw
4,ARGENTINA,SAUDI ARABIA,1.638676,-1.642469,0.039450,1,2,22 NOV 2022,11 : 00,Group C,...,14,0,0,65,80,163,361,3,-1,loss


## 4.1. Pipeline completo reutilizado desde F2

El feedback de la Formativa 3 indicó que no bastaba con transformar la lista de goles: la rúbrica solicita ejecutar el pipeline completo de preprocesamiento dentro del notebook F3.

Por ello, en esta sección se ejecuta un pipeline equivalente al desarrollado en F2, reutilizando las funciones del módulo `src/funciones_reutilizables.py` cuando está disponible. El flujo incluye:

1. Normalización de nombres de columnas.
2. Conversión de porcentajes a formato numérico.
3. Transformación básica de fecha y hora cuando existen las columnas.
4. Creación de variables derivadas del resultado.
5. Imputación de valores nulos numéricos.
6. Codificación One-Hot de variables categóricas.
7. Escalamiento z-score de variables numéricas continuas.
8. Validación final: sin nulos, sin columnas no numéricas y con dimensiones válidas.

Se conserva un DataFrame algorítmico (`df`) para el cálculo interpretable de goles y se genera un DataFrame modelable (`df_pipeline_f2_completo`) que evidencia el pipeline completo solicitado.


In [4]:
def transformar_fecha_hora_segura(df: pd.DataFrame) -> pd.DataFrame:
    """Crea variables temporales derivadas si existen columnas de fecha y hora.

    La función es tolerante a formatos distintos: si alguna columna no existe,
    simplemente retorna el DataFrame sin fallar. Esto mantiene la ejecución
    reproducible en distintos entornos.
    """
    df = df.copy()

    if "date" in df.columns:
        fecha = pd.to_datetime(df["date"], errors="coerce")
        if fecha.notna().any():
            df["match_day"] = fecha.dt.day.fillna(0).astype(int)
            df["match_month"] = fecha.dt.month.fillna(0).astype(int)
        df = df.drop(columns=["date"])

    if "hour" in df.columns:
        hora = (
            df["hour"]
            .astype(str)
            .str.replace(" ", "", regex=False)
            .str.split(":")
            .str[0]
        )
        df["match_hour"] = pd.to_numeric(hora, errors="coerce").fillna(0).astype(int)
        df = df.drop(columns=["hour"])

    return df


def codificar_categoricas(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """Codifica variables categóricas con One-Hot Encoding.

    Decisión técnica: se aplica One-Hot Encoding porque las categorías son nominales
    y no tienen un orden natural. Esto evita imponer jerarquías artificiales.
    """
    df = df.copy()
    grupos_one_hot: dict[str, list[str]] = {}

    columnas_categoricas = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    for columna in columnas_categoricas:
        if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "codificar_one_hot_auto"):
            df, nuevas = f2_utils.codificar_one_hot_auto(df, columna)
        else:
            dummies = pd.get_dummies(df[columna], prefix=columna.replace(" ", "_"), dtype=int)
            nuevas = dummies.columns.tolist()
            df = pd.concat([df.drop(columns=[columna]), dummies], axis=1)

        grupos_one_hot[columna] = nuevas

    return df, grupos_one_hot


def columnas_continuas_para_escalar(df: pd.DataFrame, excluir: list[str] | None = None) -> list[str]:
    """Selecciona columnas numéricas continuas, excluyendo binarias y variables objetivo."""
    excluir = excluir or []
    columnas = []

    for col in df.select_dtypes(include=[np.number]).columns:
        if col in excluir:
            continue
        valores = set(pd.Series(df[col].dropna().unique()).astype(float).tolist())
        es_binaria = valores.issubset({0.0, 1.0})
        if not es_binaria:
            columnas.append(col)

    return columnas


def validar_pipeline_completo(df: pd.DataFrame) -> bool:
    """Valida que el pipeline completo deje datos consistentes para análisis/modelado."""
    total_nulos = int(df.isnull().sum().sum())
    no_numericas = df.select_dtypes(exclude=[np.number]).columns.tolist()

    assert df.shape[0] > 0, "El dataset final quedó sin filas."
    assert df.shape[1] > 0, "El dataset final quedó sin columnas."
    assert total_nulos == 0, f"Quedan {total_nulos} valores nulos."
    assert not no_numericas, f"Quedan columnas no numéricas: {no_numericas}"

    print("[OK] Pipeline completo validado")
    print(f"[OK] Filas: {df.shape[0]}")
    print(f"[OK] Columnas finales: {df.shape[1]}")
    print(f"[OK] Valores nulos: {total_nulos}")
    print("[OK] Todas las columnas finales son numéricas")
    return True


def ejecutar_pipeline_f2_completo(df_entrada: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, list[str]]]:
    """Ejecuta el pipeline completo reutilizado de F2 dentro del notebook F3.

    Retorna
    -------
    tuple
        DataFrame final modelable, reporte de etapas y grupos one-hot creados.
    """
    reporte = []
    df_work = df_entrada.copy()
    reporte.append(("entrada", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 1. Normalización de nombres de columnas.
    if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "normalizar_nombres_columnas"):
        df_work = f2_utils.normalizar_nombres_columnas(df_work)
    else:
        df_work = normalizar_nombres_columnas(df_work)
    reporte.append(("normalizacion_columnas", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 2. Casting de porcentajes.
    columnas_porcentaje = [
        "possession team1",
        "possession team2",
        "possession in contest"
    ]
    for col in columnas_porcentaje:
        if col in df_work.columns:
            if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "convertir_porcentaje"):
                df_work = f2_utils.convertir_porcentaje(df_work, col)
            else:
                df_work = convertir_porcentaje(df_work, col)
    reporte.append(("casting_porcentajes", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 3. Fecha/hora a variables numéricas derivadas.
    df_work = transformar_fecha_hora_segura(df_work)
    reporte.append(("transformacion_fecha_hora", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 4. Variables derivadas del resultado.
    if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "crear_variables_resultado"):
        df_work = f2_utils.crear_variables_resultado(df_work)
        # La versión del módulo F2 crea diferencia y resultado; F3 agrega total_goals para análisis.
        if "total_goals" not in df_work.columns:
            df_work["total_goals"] = df_work["number of goals team1"] + df_work["number of goals team2"]
    else:
        df_work = crear_variables_resultado(df_work)
    reporte.append(("variables_resultado", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 5. Imputación de nulos numéricos.
    numericas = df_work.select_dtypes(include=[np.number]).columns.tolist()
    for col in numericas:
        if df_work[col].isnull().any():
            if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "imputar_nulos_numericos"):
                df_work = f2_utils.imputar_nulos_numericos(df_work, col, estrategia="mediana")
            else:
                df_work[col] = df_work[col].fillna(df_work[col].median())
    reporte.append(("imputacion_numerica", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 6. Codificación de variables categóricas.
    df_work, grupos_one_hot = codificar_categoricas(df_work)
    reporte.append(("one_hot_categoricas", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 7. Escalamiento de variables numéricas continuas.
    excluir_escalamiento = [
        "number of goals team1",
        "number of goals team2",
        "total_goals",
        "goal_difference_team1",
    ]
    columnas_escalar = columnas_continuas_para_escalar(df_work, excluir=excluir_escalamiento)

    if columnas_escalar:
        if MODULOS_REPO_DISPONIBLES and hasattr(f2_utils, "escalar_caracteristicas"):
            df_work, _ = f2_utils.escalar_caracteristicas(df_work, columnas_escalar)
        else:
            df_work = escalar_zscore(df_work, columnas_escalar)
    reporte.append(("escalamiento_zscore", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    # 8. Validación final.
    validar_pipeline_completo(df_work)
    reporte.append(("validacion_final", df_work.shape[0], df_work.shape[1], int(df_work.isnull().sum().sum())))

    reporte_df = pd.DataFrame(
        reporte,
        columns=["etapa", "filas", "columnas", "valores_nulos"]
    )

    return df_work, reporte_df, grupos_one_hot


df_pipeline_f2_completo, reporte_pipeline_f2, grupos_one_hot_f2 = ejecutar_pipeline_f2_completo(df_original)

print("\nResumen de etapas del pipeline completo reutilizado de F2:")
display(reporte_pipeline_f2)

print("\nPrimeras columnas del dataset final modelable:")
display(df_pipeline_f2_completo.head())


'team1' codificada en 32 columnas.
'team2' codificada en 32 columnas.
'category' codificada en 13 columnas.
'result_team1' codificada en 3 columnas.
[OK] Pipeline completo validado
[OK] Filas: 64
[OK] Columnas finales: 168
[OK] Valores nulos: 0
[OK] Todas las columnas finales son numéricas

Resumen de etapas del pipeline completo reutilizado de F2:


,etapa,filas,columnas,valores_nulos
0,entrada,64,88,0
1,normalizacion_columnas,64,88,0
2,casting_porcentajes,64,88,0
3,transformacion_fecha_hora,64,89,0
4,variables_resultado,64,92,0
5,imputacion_numerica,64,92,0
6,one_hot_categoricas,64,168,0
7,escalamiento_zscore,64,168,0
8,validacion_final,64,168,0



Primeras columnas del dataset final modelable:


,possession team1,possession team2,possession in contest,number of goals team1,number of goals team2,total attempts team1,total attempts team2,conceded team1,conceded team2,goal inside the penalty area team1,...,category_Group F,category_Group G,category_Group H,category_Play-off for third place,category_Quarter-final,category_Round of 16,category_Semi-final,result_team1_draw,result_team1_loss,result_team1_win
0,-0.222014,0.543986,-1.643741,0,2,-1.244675,-0.916545,0.850178,-1.025341,-0.947034,...,0,0,0,0,0,0,0,0,1,0
1,2.315290,-2.062941,-1.222943,6,2,0.376886,-0.569451,0.850178,2.872985,2.921701,...,0,0,0,0,0,0,0,0,0,1
2,-0.052861,0.123514,-0.381348,0,2,0.579581,-0.395904,0.850178,-1.025341,-0.947034,...,0,0,0,0,0,0,0,0,1,0
3,0.539177,-0.381053,-0.802146,1,1,-1.041980,-0.742998,-0.104408,-0.375620,-0.302245,...,0,0,0,0,0,0,0,1,0,0
4,1.638676,-1.642469,0.039450,1,2,0.579581,-1.437185,0.850178,-0.375620,-0.302245,...,0,0,0,0,0,0,0,0,1,0


### Interpretación del pipeline completo

El pipeline completo deja un dataset final sin valores nulos y con variables numéricas, después de aplicar casting, codificación categórica y escalamiento. Esta sección corrige explícitamente la observación del profesor: en F3 ya no se trabaja solo con la transformación de goles, sino que se ejecuta el flujo completo de preprocesamiento reutilizado desde F2 antes de desarrollar el núcleo algorítmico.

Para mantener interpretabilidad deportiva, el núcleo algorítmico de goles usa `df`, que conserva columnas como `team1` y `team2`; mientras que `df_pipeline_f2_completo` evidencia el dataset procesado completo para análisis/modelado.


## 5. Validación del dataset preprocesado

Se valida que el dataset resultante tenga las columnas mínimas requeridas, que no existan nulos numéricos luego de la imputación y que la lista de goles sea consistente con el número de partidos.


In [5]:
def validar_dataset_f3(df: pd.DataFrame, goles: list[int]) -> bool:
    """Valida condiciones mínimas del dataset para el núcleo algorítmico."""
    columnas_requeridas = [
        "team1", "team2", "number of goals team1", "number of goals team2",
        "total_goals", "goal_difference_team1", "result_team1"
    ]
    faltantes = [c for c in columnas_requeridas if c not in df.columns]
    assert not faltantes, f"Columnas requeridas faltantes: {faltantes}"
    assert len(df) > 0, "El DataFrame no puede estar vacío"
    assert len(goles) == len(df) * 2, "La lista de goles debe contener dos valores por partido"
    assert df.select_dtypes(include=[np.number]).isnull().sum().sum() == 0, "Existen nulos numéricos"
    assert (df[["number of goals team1", "number of goals team2"]] >= 0).all().all(), "Los goles no pueden ser negativos"
    return True

validar_dataset_f3(df, goles)
print("[OK] Dataset validado para Fase 3")


[OK] Dataset validado para Fase 3


## 6. Núcleo algorítmico estructurado

El núcleo algorítmico se centra en resolver una misma tarea —sumar los goles registrados— mediante tres estrategias. Esto permite comparar legibilidad, diseño y eficiencia.


In [6]:
def suma_iterativa(valores: list[int]) -> int:
    """Suma una lista de valores usando acumulador e iteración.

    Complejidad temporal: O(n), porque recorre todos los elementos una vez.
    Complejidad espacial: O(1), porque solo usa una variable acumuladora.
    """
    total = 0
    for valor in valores:
        total += valor
    return total


def suma_recursiva(valores: list[int]) -> int:
    """Suma una lista de valores usando recursividad.

    Caso base: lista vacía.
    Caso recursivo: primer elemento + suma del resto.

    Complejidad temporal: O(n).
    Complejidad espacial: O(n), por la pila de llamadas recursivas.
    """
    if len(valores) == 0:
        return 0
    return valores[0] + suma_recursiva(valores[1:])


def suma_vectorizada_pandas(df: pd.DataFrame) -> int:
    """Suma goles usando operaciones vectorizadas de pandas.

    Complejidad temporal teórica: O(n), pero con menor sobrecarga interpretada
    que un bucle Python puro en datasets grandes.
    """
    return int(df[["number of goals team1", "number of goals team2"]].sum().sum())


## 7. Algoritmo iterativo

La solución iterativa usa una estructura `for` y un acumulador. Es directa, fácil de leer y mantiene bajo consumo de memoria.


In [7]:
resultado_iterativo = suma_iterativa(goles)
print("Total de goles con algoritmo iterativo:", resultado_iterativo)


Total de goles con algoritmo iterativo: 172


## 8. Algoritmo recursivo

La solución recursiva permite evidenciar el patrón caso base + caso recursivo. Aunque no es la opción más eficiente en memoria para este problema, cumple un rol formativo importante para mostrar diseño algorítmico.


In [8]:
resultado_recursivo = suma_recursiva(goles)
print("Total de goles con algoritmo recursivo:", resultado_recursivo)


Total de goles con algoritmo recursivo: 172


## 9. Algoritmo vectorizado con pandas

La solución vectorizada aprovecha operaciones internas de pandas, evitando recorrer fila por fila desde Python.


In [9]:
resultado_vectorizado = suma_vectorizada_pandas(df)
print("Total de goles con algoritmo vectorizado:", resultado_vectorizado)


Total de goles con algoritmo vectorizado: 172


## 10. Validación técnica del código

Se prueban tres escenarios solicitados por la guía: caso normal, caso límite y excepciones. Las validaciones quedan como evidencia reproducible dentro del notebook.


In [10]:
# Caso normal: las tres implementaciones deben entregar el mismo resultado.
assert resultado_iterativo == resultado_recursivo == resultado_vectorizado
print("[OK] Caso normal: los tres algoritmos entregan el mismo total")

# Caso límite: lista vacía debe sumar cero.
assert suma_iterativa([]) == 0
assert suma_recursiva([]) == 0
print("[OK] Caso límite: lista vacía validada")

# Caso límite: un solo elemento debe devolver el mismo valor.
assert suma_iterativa([5]) == 5
assert suma_recursiva([5]) == 5
print("[OK] Caso límite: lista con un elemento validada")

# Excepción: columna inexistente en convertir_porcentaje.
try:
    convertir_porcentaje(pd.DataFrame({"a": ["10%"]}), "posesion")
except KeyError as e:
    print("[OK] Excepción capturada en convertir_porcentaje:", e)

# Excepción: estrategia inválida de imputación.
try:
    imputar_nulos_numericos(pd.DataFrame({"x": [1, np.nan]}), estrategia="moda")
except ValueError as e:
    print("[OK] Excepción capturada en imputar_nulos_numericos:", e)

# Excepción: escalamiento sobre columna no numérica.
try:
    escalar_zscore(pd.DataFrame({"texto": ["a", "b"]}), ["texto"])
except TypeError as e:
    print("[OK] Excepción capturada en escalar_zscore:", e)


[OK] Caso normal: los tres algoritmos entregan el mismo total
[OK] Caso límite: lista vacía validada
[OK] Caso límite: lista con un elemento validada
[OK] Excepción capturada en convertir_porcentaje: "La columna 'posesion' no existe en el DataFrame"
[OK] Excepción capturada en imputar_nulos_numericos: La estrategia debe ser 'media' o 'mediana'
[OK] Excepción capturada en escalar_zscore: La columna 'texto' debe ser numérica para escalar


## 11. Medición de eficiencia y análisis Big-O

Se mide la misma tarea con `timeit`: sumar goles mediante iteración, recursividad y operación vectorizada. La comparación se interpreta en términos de complejidad temporal y costo práctico.


In [11]:
# Para medir de forma estable, se replica la lista de goles.
# La versión recursiva se mide con una muestra acotada para no superar
# el límite de recursión de Python en listas demasiado grandes.
goles_medicion = goles * 1000
goles_medicion_recursivo = goles * 20

df_medicion = pd.DataFrame({
    "number of goals team1": goles_medicion[0::2],
    "number of goals team2": goles_medicion[1::2]
})

repeticiones = 100

tiempo_iterativo = timeit.timeit(lambda: suma_iterativa(goles_medicion), number=repeticiones)
tiempo_recursivo = timeit.timeit(lambda: suma_recursiva(goles_medicion_recursivo), number=repeticiones)
tiempo_vectorizado = timeit.timeit(lambda: suma_vectorizada_pandas(df_medicion), number=repeticiones)

resultados_tiempo = pd.DataFrame({
    "algoritmo": ["iterativo", "recursivo", "vectorizado_pandas"],
    "tamano_muestra": [len(goles_medicion), len(goles_medicion_recursivo), len(goles_medicion)],
    "tiempo_segundos": [tiempo_iterativo, tiempo_recursivo, tiempo_vectorizado],
    "big_o_temporal": ["O(n)", "O(n)", "O(n)"],
    "big_o_espacial": ["O(1)", "O(n)", "O(1) aprox."]
}).sort_values("tiempo_segundos")

resultados_tiempo

,algoritmo,tamano_muestra,tiempo_segundos,big_o_temporal,big_o_espacial
2,vectorizado_pandas,128000,0.020668,O(n),O(1) aprox.
0,iterativo,128000,0.294328,O(n),O(1)
1,recursivo,2560,1.212006,O(n),O(n)


### Interpretación de eficiencia

Las tres estrategias tienen complejidad temporal **O(n)**, porque deben considerar todos los registros de goles. Sin embargo, difieren en costo práctico:

- La versión **iterativa** recorre la lista con bajo uso de memoria.
- La versión **recursiva** también recorre todos los elementos, pero usa pila de llamadas, por lo que su complejidad espacial es mayor.
- La versión **vectorizada** delega operaciones a pandas, lo que suele ser más conveniente en flujos de ciencia de datos porque evita bucles explícitos en Python y se integra mejor con el DataFrame.


## 12. Diseño estructurado, cohesión y bajo acoplamiento

La arquitectura se separa en componentes pequeños y reutilizables:

| Componente | Responsabilidad | Criterio técnico |
|---|---|---|
| `cargar_datos()` | Entrada de datos | Bajo acoplamiento: solo carga el dataset |
| `preprocesar_datos()` | Orquestar limpieza y transformación | Alta cohesión: agrupa el pipeline de F2 |
| `preparar_lista_goles()` | Preparar entrada algorítmica | Separa datos crudos del análisis |
| `suma_iterativa()` | Algoritmo estructurado | Una tarea específica |
| `suma_recursiva()` | Algoritmo recursivo | Caso base y caso recursivo claros |
| `suma_vectorizada_pandas()` | Alternativa optimizada | Usa operación vectorizada |
| `validar_dataset_f3()` | Validación técnica | Evidencia reproducible |

Este diseño evita una celda gigante con toda la lógica mezclada. Cada función puede probarse de manera individual, lo que mejora la mantenibilidad y la trazabilidad.


## 13. Algoritmo recursivo adicional: Merge Sort

Además de la suma recursiva, se implementa `merge sort` para ordenar los partidos según su cantidad total de goles. Este algoritmo usa divide y vencerás y tiene complejidad **O(n log n)**.


In [12]:
def combinar_ordenado(izquierda: list[int], derecha: list[int]) -> list[int]:
    """Combina dos listas ordenadas en una sola lista ordenada."""
    resultado = []
    i = j = 0

    while i < len(izquierda) and j < len(derecha):
        if izquierda[i] <= derecha[j]:
            resultado.append(izquierda[i])
            i += 1
        else:
            resultado.append(derecha[j])
            j += 1

    resultado.extend(izquierda[i:])
    resultado.extend(derecha[j:])
    return resultado


def merge_sort(lista: list[int]) -> list[int]:
    """Ordena una lista mediante Merge Sort recursivo.

    Complejidad temporal: O(n log n).
    Complejidad espacial: O(n).
    """
    if len(lista) <= 1:
        return lista

    medio = len(lista) // 2
    izquierda = merge_sort(lista[:medio])
    derecha = merge_sort(lista[medio:])
    return combinar_ordenado(izquierda, derecha)


totales_goles_partido = df["total_goals"].astype(int).tolist()
totales_ordenados = merge_sort(totales_goles_partido)

print("Totales de goles por partido:", totales_goles_partido)
print("Totales ordenados con merge sort:", totales_ordenados)
assert totales_ordenados == sorted(totales_goles_partido)
print("[OK] Merge Sort validado contra sorted()")


Totales de goles por partido: [2, 8, 2, 2, 3, 0, 0, 5, 0, 3, 7, 1, 1, 0, 5, 2, 2, 4, 2, 0, 1, 2, 3, 2, 1, 2, 5, 2, 6, 5, 1, 2, 2, 3, 3, 1, 1, 1, 2, 3, 0, 3, 3, 6, 2, 3, 5, 1, 4, 3, 4, 3, 2, 5, 0, 7, 2, 4, 1, 3, 3, 2, 3, 6]
Totales ordenados con merge sort: [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 6, 6, 6, 7, 7, 8]
[OK] Merge Sort validado contra sorted()


## 14. Programación Orientada a Objetos

La guía de Sumativa 3 solicita decidir entre justificar o implementar POO. En esta versión se opta por **implementar POO directamente**, para cubrir el criterio completo.

La implementación considera:

- **Encapsulamiento:** clases con estado interno protegido mediante atributos como `_df` y `_goles`.
- **Herencia:** clases concretas que heredan de contratos abstractos (`Transformador`, `CalculadorGoles`).
- **Polimorfismo:** distintas clases responden al mismo método (`aplicar()` o `calcular()`), permitiendo ejecutar etapas y algoritmos de forma intercambiable.
- **Composición:** la clase `Pipeline` recibe una lista de etapas y las ejecuta en orden, sin depender de su implementación interna.

Esta decisión responde al feedback de la Formativa 3 y fortalece la relación F2 → F3: el pipeline de F2 se encapsula y se conecta con el núcleo algorítmico de F3.


In [13]:
class PreprocesadorFIFA:
    """Encapsula pasos de preprocesamiento del dataset FIFA.

    El atributo _df representa estado interno protegido. Los métodos devuelven
    self para permitir encadenamiento de operaciones.
    """

    def __init__(self, df: pd.DataFrame):
        self._df = df.copy()

    def normalizar_columnas(self):
        self._df = normalizar_nombres_columnas(self._df)
        return self

    def convertir_porcentajes(self):
        for col in ["possession team1", "possession team2", "possession in contest"]:
            if col in self._df.columns:
                self._df = convertir_porcentaje(self._df, col)
        return self

    def crear_resultado(self):
        self._df = crear_variables_resultado(self._df)
        return self

    def imputar_numericas(self, estrategia: str = "mediana"):
        self._df = imputar_nulos_numericos(self._df, estrategia)
        return self

    def escalar(self, columnas: list[str]):
        columnas_existentes = [c for c in columnas if c in self._df.columns]
        self._df = escalar_zscore(self._df, columnas_existentes)
        return self

    def resultado(self) -> pd.DataFrame:
        return self._df.copy()


class Transformador(ABC):
    """Contrato común para etapas de transformación del pipeline."""

    @abstractmethod
    def aplicar(self, df: pd.DataFrame) -> pd.DataFrame:
        pass


class CreadorResultado(Transformador):
    """Crea variables derivadas del resultado del partido."""

    def aplicar(self, df: pd.DataFrame) -> pd.DataFrame:
        return crear_variables_resultado(df)


class ImputadorMediana(Transformador):
    """Imputa nulos numéricos usando mediana."""

    def aplicar(self, df: pd.DataFrame) -> pd.DataFrame:
        return imputar_nulos_numericos(df, estrategia="mediana")


class EscaladorZScore(Transformador):
    """Escala columnas numéricas con z-score."""

    def __init__(self, columnas: list[str]):
        self.columnas = columnas

    def aplicar(self, df: pd.DataFrame) -> pd.DataFrame:
        columnas_existentes = [c for c in self.columnas if c in df.columns]
        return escalar_zscore(df, columnas_existentes)


class Pipeline:
    """Orquesta etapas intercambiables mediante composición y polimorfismo."""

    def __init__(self, etapas: list[Transformador]):
        self.etapas = etapas

    def ejecutar(self, df: pd.DataFrame) -> pd.DataFrame:
        resultado = normalizar_nombres_columnas(df)
        for col in ["possession team1", "possession team2", "possession in contest"]:
            if col in resultado.columns:
                resultado = convertir_porcentaje(resultado, col)
        for etapa in self.etapas:
            resultado = etapa.aplicar(resultado)
        return resultado


In [14]:
class CalculadorGoles(ABC):
    """Clase abstracta para calcular el total de goles."""

    @abstractmethod
    def calcular(self) -> int:
        pass


class CalculadorIterativo(CalculadorGoles):
    """Calcula goles usando algoritmo iterativo."""

    def __init__(self, goles: list[int]):
        self._goles = goles

    def calcular(self) -> int:
        return suma_iterativa(self._goles)


class CalculadorRecursivo(CalculadorGoles):
    """Calcula goles usando algoritmo recursivo."""

    def __init__(self, goles: list[int]):
        self._goles = goles

    def calcular(self) -> int:
        return suma_recursiva(self._goles)


class CalculadorVectorizado(CalculadorGoles):
    """Calcula goles usando operación vectorizada de pandas."""

    def __init__(self, df: pd.DataFrame):
        self._df = df.copy()

    def calcular(self) -> int:
        return suma_vectorizada_pandas(self._df)


## 15. Pipeline POO con herencia, polimorfismo y encapsulamiento

Se ejecuta el pipeline orientado a objetos. El mismo método `aplicar()` funciona para distintas clases, lo que evidencia polimorfismo. El objeto `Pipeline` no necesita conocer la implementación interna de cada etapa.


In [15]:
columnas_escalamiento = [
    "possession team1", "possession team2", "possession in contest",
    "total attempts team1", "total attempts team2",
    "passes team1", "passes team2"
]

pipeline_poo = Pipeline([
    CreadorResultado(),
    ImputadorMediana(),
    EscaladorZScore(columnas_escalamiento)
])

df_poo = pipeline_poo.ejecutar(df_original)
goles_poo = preparar_lista_goles(df_poo)

calculadores = [
    CalculadorIterativo(goles_poo),
    CalculadorRecursivo(goles_poo),
    CalculadorVectorizado(df_poo)
]

for calculador in calculadores:
    print(type(calculador).__name__, "=>", calculador.calcular())

assert all(c.calcular() == resultado_vectorizado for c in calculadores)
print("[OK] Polimorfismo validado: todos los calculadores entregan el mismo total")


CalculadorIterativo => 172
CalculadorRecursivo => 172
CalculadorVectorizado => 172
[OK] Polimorfismo validado: todos los calculadores entregan el mismo total


## 16. Detección algorítmica de partidos destacados

Como extensión del núcleo algorítmico, se detectan partidos con alta cantidad de goles mediante una regla simple basada en el percentil 75. Esto conecta el algoritmo con la problemática deportiva: identificar partidos ofensivamente destacados.


In [16]:
def detectar_partidos_destacados(df: pd.DataFrame, columna: str = "total_goals") -> pd.DataFrame:
    """Detecta partidos cuyo total de goles está sobre o igual al percentil 75."""
    if columna not in df.columns:
        raise KeyError(f"No existe la columna '{columna}'")
    umbral = df[columna].quantile(0.75)
    columnas_salida = ["team1", "team2", "number of goals team1", "number of goals team2", columna]
    columnas_salida = [c for c in columnas_salida if c in df.columns]
    return df.loc[df[columna] >= umbral, columnas_salida].sort_values(columna, ascending=False)

partidos_destacados = detectar_partidos_destacados(df)
print("Partidos destacados por total de goles:")
partidos_destacados


Partidos destacados por total de goles:


,team1,team2,number of goals team1,number of goals team2,total_goals
1,ENGLAND,IRAN,6,2,8
10,SPAIN,COSTA RICA,7,0,7
55,PORTUGAL,SWITZERLAND,6,1,7
28,CAMEROON,SERBIA,3,3,6
43,COSTA RICA,GERMANY,2,4,6
63,ARGENTINA,FRANCE,3,3,6
7,FRANCE,AUSTRALIA,4,1,5
14,PORTUGAL,GHANA,3,2,5
26,CROATIA,CANADA,4,1,5
29,KOREA REPUBLIC,GHANA,2,3,5


## 17. Evidencia final de ejecución reproducible

Esta sección resume las salidas principales del notebook y deja evidencia de que el flujo completo puede ejecutarse de inicio a fin.


In [17]:
resumen_final = pd.DataFrame({
    "indicador": [
        "filas_dataset_original",
        "columnas_dataset_original",
        "filas_dataset_preprocesado_algoritmico",
        "columnas_dataset_preprocesado_algoritmico",
        "filas_pipeline_f2_completo",
        "columnas_pipeline_f2_completo",
        "total_goles",
        "partidos_destacados",
        "algoritmos_validados",
        "poo_validada",
        "pipeline_f2_validado"
    ],
    "valor": [
        df_original.shape[0],
        df_original.shape[1],
        df.shape[0],
        df.shape[1],
        df_pipeline_f2_completo.shape[0],
        df_pipeline_f2_completo.shape[1],
        resultado_vectorizado,
        len(partidos_destacados),
        "iterativo, recursivo, vectorizado, merge sort",
        "encapsulamiento, herencia, polimorfismo",
        "NA, casting, one-hot, escalamiento, validacion"
    ]
})

display(resumen_final)


,indicador,valor
0,filas_dataset_original,64
1,columnas_dataset_original,88
2,filas_dataset_preprocesado_algoritmico,64
3,columnas_dataset_preprocesado_algoritmico,91
4,filas_pipeline_f2_completo,64
5,columnas_pipeline_f2_completo,168
6,total_goles,172
7,partidos_destacados,16
8,algoritmos_validados,"iterativo, recursivo, vectorizado, merge sort"
9,poo_validada,"encapsulamiento, herencia, polimorfismo"


## 18. Instrucciones para repositorio GitHub

Para mantener la coherencia con la estructura real del proyecto y favorecer la reproducibilidad, se recomienda conservar la siguiente organización:

```text
fifa-worldcup-2022-analysis/
├── data/
│   ├── raw/
│   │   └── Fifa_world_cup_matches.csv
│   └── processed/
│       └── fifa_world_cup_matches_processed.csv
├── docs/
├── notebooks/
│   ├── F1_Definicion.ipynb
│   ├── F2_Definicion.ipynb
│   ├── F3_Definicion.ipynb
│  
├── results/
│   ├── figures/
│   │   └── .gitkeep
│   └── reports/
│       └── .gitkeep
├── src/
│   ├── funciones_reutilizables.py
│   └── funciones_f3_algoritmos.py
├── README.md
├── requirements.txt
├── .gitignore
└── .mailmap
```

Los notebooks de F1, F2 y F3 se almacenan en `notebooks/`, mientras que la lógica reutilizable se concentra en `src/`. Para la Sumativa 3 deben quedar especialmente actualizados:

- `src/funciones_reutilizables.py`
- `src/funciones_f3_algoritmos.py`
- `README.md`
- `requirements.txt`

Además, se recomienda ejecutar `Kernel → Restart & Run All` antes del último commit y dejar evidencia de commits trazables por integrante.


## 19. Conclusiones

La Fase 3 permitió transformar el trabajo de limpieza y preparación de datos de la Fase 2 en una solución algorítmica más completa. Se implementaron funciones con responsabilidades claras, validaciones técnicas, mediciones reproducibles con `timeit`, recursividad y una arquitectura orientada a objetos.

En respuesta al feedback de la Formativa 3, esta versión incorpora explícitamente el pipeline completo reutilizado desde F2 dentro del notebook F3. El flujo incluye normalización de columnas, casting de porcentajes, manejo de nulos, creación de variables derivadas, codificación categórica, escalamiento y validación final.

También se decidió implementar POO directamente, mediante clases con encapsulamiento, herencia, polimorfismo y composición. Esto permite conectar el preprocesamiento de F2 con el núcleo algorítmico de F3 de manera más mantenible y extensible.

Los enfoques de cálculo de goles entregan resultados consistentes, lo que valida la lógica principal. La comparación de eficiencia muestra que, aunque varias estrategias tienen complejidad temporal lineal, las operaciones vectorizadas con pandas presentan mejor rendimiento práctico por sus optimizaciones internas. El algoritmo recursivo `merge sort` complementa el análisis al evidenciar una estrategia divide and conquer con complejidad O(n log n).


## 20. Bibliografía

- Downey, A. B. (2015). *Think Python: How to Think Like a Computer Scientist* (2nd ed.). Green Tea Press. https://greenteapress.com/wp/think-python-2e/
- McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media.
- NumPy Developers. (2024). *NumPy documentation*. https://numpy.org/doc/
- pandas development team. (2024). *pandas documentation*. https://pandas.pydata.org/docs/
- Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.

### Citas sugeridas dentro del informe

- Para diseño modular: *(Downey, 2015; McKinney, 2022).*  
- Para operaciones vectorizadas con pandas: *(pandas development team, 2024; McKinney, 2022).*  
- Para arreglos numéricos y eficiencia computacional: *(NumPy Developers, 2024).*  
- Para normalización o preprocesamiento en ciencia de datos: *(Pedregosa et al., 2011).*


## 21. Plan de integración mediante Pull Requests

La construcción del notebook final se organiza en tres contribuciones incrementales sobre un único archivo: `notebooks/F3_Definicion.ipynb`.

| Integrante | Alcance | Rama sugerida | Commit convencional |
|---|---|---|---|
| Claudio Alarcón | Secciones 1 a 4: configuración, carga de datos y pipeline completo reutilizado desde F2 | `feature/integrar_pipeline` | `feat(f3): integrar pipeline completo reutilizado desde F2` |
| Enzo Pinilla | Secciones 1 a 12: núcleo algorítmico, validaciones, `timeit` y complejidad | `feature/integrar_nucleo` | `feat(f3): implementar nucleo algoritmico y mediciones de complejidad` |
| Luis Rodrigo Espinoza | Secciones 1 a 22: POO, documentación, repositorio, conclusiones y cierre final | `feature/incorporar_poo` | `feat(f3): incorporar POO y documentacion final de arquitectura` |

Este flujo permite que cada Pull Request agregue valor sobre el anterior, manteniendo un único notebook final y un historial de Git claro, trazable y coherente con la rúbrica.

## Prueba rama feature/f4-algoritmos-poo